# **Calculate the MFI (Mean Fluorescence Intensity)**
In order to determine the phenotypes of each segmented cell, we compute the mean intensity of each phenotypic marker for every cell. The arcsinh transformation with a cofactor is applied to the MFIs, then the batch effect is corrected using the ComBat algorithm, and finally the data are standardized

## 🛠️ **Importation librairies et fonctions**

In [ ]:
%pip install scanpy
%pip install anndata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.2/174.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 129.0 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

import tifffile as tiff
from skimage.measure import regionprops_table

from tqdm.auto import tqdm

from scipy.stats import f_oneway
import anndata as ad
import scanpy as sc
import cv2
import skimage as ski
from skimage import measure
from pathlib import Path
from PIL import Image
from matplotlib import cm
from IPython.display import clear_output
from google.colab import drive
import matplotlib
from sklearn.preprocessing import StandardScaler
import sklearn as skl
import seaborn as sns
drive.mount('/content/gdrive')
path="/content/gdrive/MyDrive/these/"


Mounted at /content/gdrive


In [ ]:
def normalize(img):
  img=(img-np.min(img))/(np.max(img)-np.min(img))
  return img
def normalize_255(img):
  img=((img-np.min(img))/(np.max(img)-np.min(img)))*255
  return img

## 📂 **Creation of folders**
⚠️ USER INPUT REQUIRED  
*Always run this block*

From the marker images for each ROI of the slide, the MFI of the pixels within each cell is computed using the CSV file

In [ ]:
import os
path = "/content/gdrive/MyDrive/"
bool_exist=False
while bool_exist==False:
 name_path = input("Enter the path where the project has been be created (if the project is in 'My Drive' just press 'Enter): ")
 name_project = input("Enter the name of the project: ")
 if name_path!="":
  path += name_path + "/pipeline/"+name_project+"/"
 else:
  path+="pipeline/"+name_project+"/"
 bool_exist=os.path.isdir(path)
 if bool_exist==False:
   print(f"❌ The path {path} does not exist")
 else:
  print("✅ Project :"+path)
  bool_exist=True

path_img_raw=path+"images/raw/"
path_mask_cell=path+"segmentation/segmentation_cells/mask_filtered/"

if os.path.isdir(path_img_raw)==False:
   print("❌ You have not completed the first step of creating PNG images")
elif os.path.isdir(path_mask_cell)==False:
   print("❌ You have not completed the first step of cells segmentation")
else:
 path_clust=path+"clustering/"
 if os.path.isdir(path_clust)==False:
     os.mkdir(path_clust)
     print("✅ Folder for clustering created")
 path_clust_mfi=path_clust+"MFI/"
 if os.path.isdir(path_clust_mfi)==False:
     os.mkdir(path_clust_mfi)
     print("✅ Folder for MFI created")
 path_clust_mfi_raw=path_clust_mfi+"Raw/"
 if os.path.isdir(path_clust_mfi_raw)==False:
     os.mkdir(path_clust_mfi_raw)
     print("✅ Folder for MFI Raw created")
 path_segmentation_df=path+"segmentation/segmentation_cells/df_ROI/"
 if os.path.isdir(path_segmentation_df)==False:
     os.mkdir(path_segmentation_df)
     print("✅ Folder for segmentation created")
algo=input("Enter the algorithm used for segmentation: ")
if os.path.isdir(path_mask_cell+algo):
  print(f"✅ {algo} will be choosen for the calcul of MFI")
  path_mask_cell=path_mask_cell+algo+"/"
else:
  print("❌ this algorithm does not exist")


Enter the path where the project has been be created (if the project is in 'My Drive' just press 'Enter): these
Enter the name of the project: Appendix
✅ Project :/content/gdrive/MyDrive/these/pipeline/Appendix/
Enter the algorithm used for segmentation: mesmer
✅ mesmer will be choosen for the calcul of MFI


## ⚙️ **Dataframe with cells features and MFI for each ROI**
This code block creates a DataFrame for each biopsy image containing the different characteristics of the segmented cells such as size, centroid, area...

In [ ]:
LABEL_COL = "label"
SAVE_COORDS = False

IMG_EXTS = (".tif", ".tiff", ".png", ".jpg", ".jpeg")

def read_image_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".tif", ".tiff"):
        arr = tiff.imread(path)
    else:
        arr = np.array(Image.open(path))
    # (H,W,C) -> canal 0
    if arr.ndim > 2:
        arr = arr[..., 0]
    return arr

def list_image_files(folder):
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(IMG_EXTS)])

list_df_all_cells = []

list_mask_files = sorted([
    f for f in os.listdir(path_mask_cell)
    if f.lower().endswith(IMG_EXTS)
])

for mask_file in tqdm(list_mask_files, desc="Processing of the masks / ROI"):
    mask_path = os.path.join(path_mask_cell, mask_file)

    try:
        mask = read_image_any(mask_path).astype(np.int32)
    except Exception as e:
        print(f"[ERREUR] Mask not readable {mask_file} : {e}")
        continue

    if mask.max() == 0:
        continue

    roi_name = os.path.splitext(mask_file)[0]

    props = regionprops_table(
        mask,
        properties=("label", "equivalent_diameter_area", "centroid", "area"),
    )
    df_object = pd.DataFrame(props)
    df_object["ROI"] = roi_name

    if SAVE_COORDS:
        coords_col = []
        for lab in df_object[LABEL_COL].astype(int):
            ys, xs = np.where(mask == lab)
            coords_col.append(";".join(f"{x},{y}" for x, y in zip(xs, ys)))
        df_object["coord"] = coords_col

    flat_mask = mask.ravel()
    max_id = int(mask.max())

    marker_folder = os.path.join(path_img_raw, roi_name)
    if not os.path.isdir(marker_folder):
        out_path = os.path.join(path_clust_mfi_raw, roi_name + ".csv")
        df_object.to_csv(out_path, index=False)
        list_df_all_cells.append(df_object)
        continue

    marker_files = list_image_files(marker_folder)

    nonzero = counts > 0

    for file_marker in marker_files:
        img_path = os.path.join(marker_folder, file_marker)

        try:
            img = read_image_any(img_path).astype(np.float32)
        except Exception as e:
            print(f"[ERREUR] Image not readable {roi_name}/{file_marker} : {e}")
            continue

        # Sécurité taille
        if img.shape != mask.shape:
            print(f"[WARN] Different size ROI={roi_name} marker={file_marker} img={img.shape} mask={mask.shape} -> skip")
            continue

        img_flat = img.ravel()
        marker = os.path.splitext(file_marker)[0]

        sums = np.bincount(flat_mask, weights=img_flat, minlength=max_id + 1).astype(np.float64)

        means = np.zeros(max_id + 1, dtype=np.float32)
        means[nonzero] = (sums[nonzero] / counts[nonzero]).astype(np.float32)

        df_object[marker] = df_object[LABEL_COL].map(pd.Series(means))

    df_object_clean = df_object.dropna()
    out_path = os.path.join(path_clust_mfi_raw, roi_name + ".csv")
    df_object_clean.to_csv(out_path, index=False)

    list_df_all_cells.append(df_object_clean)

# Global
if list_df_all_cells:
    df_all_cells = pd.concat(list_df_all_cells, ignore_index=True)
    out_all = os.path.join(path_clust_mfi, "df_mfi_tot_raw.csv")
    df_all_cells.to_csv(out_all, index=False)
    print(f"✅ Global: {out_all} | n={len(df_all_cells)}")
else:
    print("[No data] Nothing generated.")

print(f"✅ CSV available : {path_clust_mfi_raw}")
df_all_cells.head()

Traitement des masques / ROI:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Global: /content/gdrive/MyDrive/these/pipeline/Appendix/clustering/MFI/df_mfi_tot_raw.csv | n=230080
✅ CSV disponibles dans : /content/gdrive/MyDrive/these/pipeline/Appendix/clustering/MFI/Raw/


,label,equivalent_diameter_area,centroid-0,centroid-1,area,ROI,CD163,CD20,CD21,CD3,...,CD8,DAPI,ECadh,FOXP3,GranzB,HLADR,ICOS,IFNG,PD1,PanCK
0,2,14.228320,33.534591,741.943396,159.0,ROI,575.786194,1499.415039,640.364807,1112.012573,...,1004.647827,3233.572266,530.559753,1515.628906,568.452820,1242.974854,1143.075439,661.377380,934.798767,544.169800
1,5,17.769733,35.709677,643.830645,248.0,ROI,624.669373,3047.088623,2710.504150,1278.266113,...,1140.842773,1849.943604,580.112915,1585.350830,631.133057,1937.209717,1310.737915,685.766113,990.540344,584.379028
2,6,20.867389,34.716374,987.476608,342.0,ROI,745.631592,2316.628662,1038.827515,4959.634277,...,1154.950317,2728.561523,609.432739,1666.462036,585.005859,1206.274902,1879.938599,720.807007,1514.742676,595.172485
3,7,13.110581,35.918519,153.370370,135.0,ROI,566.607422,656.674072,823.259277,1028.548096,...,848.103699,2094.288818,534.948120,1270.303711,506.696289,566.518494,1526.903687,601.185181,941.074097,586.792603
4,8,16.273716,36.475962,269.913462,208.0,ROI,732.269226,649.966370,759.653870,3897.201904,...,1039.134644,2798.312500,686.730774,2252.235596,775.225952,589.086548,1739.471191,932.865356,1794.134644,570.687500


## 📊 **Batch effect assessment**
*This code block is used to evaluate the presence of batch effects in the dataset*

In [ ]:
path_batch_effect=path_clust+"MFI/batch_effect/"
if os.path.isdir(path_batch_effect)==False:
    os.mkdir(path_batch_effect)
    print("✅ Folder for batch effect created")

✅ Folder for batch effect created


In [ ]:
df_mfi=pd.read_csv(path_clust_mfi+"df_mfi_tot_raw.csv")
df_mfi=df_mfi.drop(["area","centroid-0","centroid-1","equivalent_diameter_area"],axis=1)
df_mfi.head()

,label,ROI,CD20,CD3,CD45,DAPI,ECadh,PanCK
0,2,ROI,1499.4150,1112.0126,1511.86790,3233.5723,530.55975,544.1698
1,5,ROI,3047.0886,1278.2661,1896.57260,1849.9436,580.11290,584.3790
2,6,ROI,2316.6287,4959.6343,2543.42700,2728.5615,609.43274,595.1725
3,7,ROI,656.6741,1028.5481,470.86667,2094.2888,534.94810,586.7926
4,8,ROI,649.9664,3897.2020,1713.34620,2798.3125,686.73080,570.6875


In [ ]:

# -------------------------------
# Utility: Benjamini-Hochberg FDR
# -------------------------------
def benjamini_hochberg(pvals, alpha=0.05):
    """
    Perform Benjamini–Hochberg FDR correction.
    Returns:
      - adjusted p-values (q-values)
      - boolean array of rejections at level alpha
    """
    pvals = np.asarray(pvals)
    n = pvals.size
    order = np.argsort(pvals)
    ranks = np.arange(1, n + 1)
    sorted_p = pvals[order]
    q = sorted_p * n / ranks
    # enforce monotonicity
    q = np.minimum.accumulate(q[::-1])[::-1]
    qvals = np.empty_like(q)
    qvals[order.argsort()] = q  # invert sorting
    reject = qvals <= alpha
    return qvals, reject

# ===============================
# 1) Split features / labels
# ===============================
label_col = "ROI"
assert label_col in df_mfi.columns, "ROI column not found in df_mfi."

# Keep only numeric columns as features (exclude ROI)
feature_cols = df_mfi.columns.drop(label_col)
X = df_mfi[feature_cols].apply(pd.to_numeric, errors="coerce")
y = df_mfi[label_col].astype(str).values

# Drop rows with any NaNs in features to avoid issues downstream
mask_valid = X.notnull().all(axis=1)
X = X[mask_valid]
y = y[mask_valid]
print(f"Using {X.shape[0]} cells and {X.shape[1]} markers after dropping NaNs.")

# ===============================
# 2) Standardize features
# ===============================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===============================
# 3) PCA (2D) and UMAP (2D)
# ===============================
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_umap = reducer.fit_transform(X_scaled)

print(f"PCA variance explained (PC1, PC2): {pca.explained_variance_ratio_[0]:.2%}, {pca.explained_variance_ratio_[1]:.2%}")

# ===============================
# 5) ANOVA per marker across batches (ROI)
# ===============================
anova_rows = []
groups = pd.Series(y).unique().tolist()

for col in feature_cols:
    # Build groups of values for this marker by ROI
    samples = [X[col].values[y == g] for g in groups]
    # Keep only groups that have at least 2 observations
    samples = [s for s in samples if len(s) >= 2]
    if len(samples) >= 2:
        F, p = f_oneway(*samples)
    else:
        F, p = np.nan, np.nan
    anova_rows.append({"marker": col, "F": F, "pval": p})

df_anova = pd.DataFrame(anova_rows).sort_values("pval", na_position="last")
# FDR (Benjamini-Hochberg)
qvals, reject = benjamini_hochberg(df_anova["pval"].fillna(1.0).values, alpha=0.05)
df_anova["qval_BH"] = qvals
df_anova["significant_FDR5%"] = reject

# Optionally save
# df_anova.to_csv("anova_by_marker.csv", index=False)

# ===============================
# 6) Plots: PCA and UMAP colored by ROI
# ===============================
def _scatter_2d(ax, emb, labels, title):
    """Simple 2D scatter with one color per label."""
    unique_labels = np.unique(labels)
    for lab in unique_labels:
        sel = labels == lab
        ax.scatter(emb[sel, 0], emb[sel, 1], s=12, alpha=0.8, label=lab)
    ax.set_title(title)
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
    ax.legend(title="ROI", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

plt.figure(figsize=(6, 5))
ax = plt.gca()
_scatter_2d(ax, X_umap, y, "UMAP")
plt.tight_layout()
plt.savefig(path_batch_effect+"umap.png")
plt.show()

# ===============================
# 7) Quick boxplots (optional): MFI distribution by ROI for top ANOVA markers
# ===============================
# This helps visually confirm batch-driven shifts for the most significant markers.
top_markers = df_anova.dropna().sort_values("qval_BH")["marker"].tolist()
if top_markers:
    n = len(top_markers)
    for m in top_markers:
        plt.figure(figsize=(6, 4))
        # Build boxplot data in label order
        data = [X[m].values[y == g] for g in groups if np.sum(y == g) > 0]
        plt.boxplot(data, labels=[g for g in groups if np.sum(y == g) > 0], showfliers=False)
        plt.title(f"{m} — MFI by ROI")
        plt.xlabel("ROI (batch)")
        plt.ylabel("MFI")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.savefig(path_batch_effect+f"boxplot_{m}.png")
print("✅ Folder with reduction of dimension and ANOVA boxplot: "+path_batch_effect)


Using 230080 cells and 7 markers after dropping NaNs.


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


KeyboardInterrupt: 

## ⚙️ **Management of Batch Effect and Outliers**

### ⚙️ **Arcsinh transformation**
⚠️ USER INPUT REQUIRED  
The arcsinh transformation with a cofactor is used to stabilize variance and compress the wide dynamic range of fluorescence intensity values, making weak and strong signals more comparable. Unlike a pure logarithmic transform, it behaves linearly near zero, which preserves information from low or even slightly negative values.

In [ ]:
cofactor=int(input("Enter the cofactor for the Arcsinh transformation (5):"))

Enter the cofactor for the Arcsinh transformation (5):5


In [ ]:
df_mfi=pd.read_csv(path_clust_mfi+"df_mfi_tot_raw.csv")
df_mfi=df_mfi.drop(["area","centroid-0","centroid-1","label","equivalent_diameter_area"],axis=1)
df_mfi.head()

,ROI,CD163,CD20,CD21,CD3,CD31,CD38,CD4,CD45,CD68,CD8,DAPI,ECadh,FOXP3,GranzB,HLADR,ICOS,IFNG,PD1,PanCK
0,ROI,575.7862,1499.4150,640.3648,1112.0126,495.52830,672.49054,1521.0126,1511.86790,1280.23270,1004.6478,3233.5723,530.55975,1515.6289,568.45280,1242.97490,1143.0754,661.37740,934.79877,544.1698
1,ROI,624.6694,3047.0886,2710.5042,1278.2661,547.41940,761.12900,1589.2258,1896.57260,1323.59680,1140.8428,1849.9436,580.11290,1585.3508,631.13306,1937.20970,1310.7379,685.76610,990.54034,584.3790
2,ROI,745.6316,2316.6287,1038.8275,4959.6343,567.07020,704.88890,2027.9854,2543.42700,1157.97070,1154.9503,2728.5615,609.43274,1666.4620,585.00586,1206.27490,1879.9386,720.80700,1514.74270,595.1725
3,ROI,566.6074,656.6741,823.2593,1028.5481,419.46667,625.18520,1192.2518,470.86667,1017.65924,848.1037,2094.2888,534.94810,1270.3037,506.69630,566.51850,1526.9037,601.18520,941.07410,586.7926
4,ROI,732.2692,649.9664,759.6539,3897.2020,435.42790,700.62020,2023.7019,1713.34620,1426.25000,1039.1346,2798.3125,686.73080,2252.2356,775.22595,589.08655,1739.4712,932.86536,1794.13460,570.6875


In [ ]:
list_col=[col for col in df_mfi.columns if col!="ROI"]
df_mfi_arcsinh = df_mfi.copy()
df_mfi_arcsinh[list_col] = np.arcsinh(df_mfi[list_col] / cofactor)
df_mfi_arcsinh.to_csv(path_clust_mfi+"df_mfi_tot_corrected.csv",index=False)
print("✅ Folder with MFI transformed Arcsinh: "+path_clust_mfi+"df_mfi_tot_corrected.csv")
df_mfi_arcsinh.isna().sum().sum()

✅ Folder with MFI transformed Arcsinh: /content/gdrive/MyDrive/these/pipeline/Appendix/clustering/MFI/df_mfi_tot_corrected.csv


np.int64(0)

### ⚙️ **ComBbat algorithm (Combating Batch effect)**
⚠️ **Run this block only if the are more than 2 images in the dataset**  

The ComBat algorithm aims to reduce batch effect by modeling and adjusting for systematic technical variations across batches while preserving true biological differences.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

# Chargement
df_mfi_arcsinh = pd.read_csv(path_clust_mfi + "df_mfi_tot_arcsinh.csv")

# --- 1. Séparation batches / features ---

# Batch = ROI (nom de la biopsie / lame, etc.)
batches = df_mfi_arcsinh["ROI"].astype(str).fillna("NA").copy()

# On ne garde pour ComBat que les colonnes numériques (tous les marqueurs)
features = df_mfi_arcsinh.drop(columns=["ROI"])
features = features.apply(pd.to_numeric, errors="coerce")  # force en numérique

# Optionnel : afficher un résumé
print("Dtypes des features :")
print(features.dtypes.head())
print("Nombre de NaN par colonne (avant ComBat) :")
print(features.isna().sum().head())

# Vérifs minimales
assert batches.nunique() > 1, "Il faut au moins 2 batches (ROI différents) pour appliquer ComBat."
assert len(features) == len(batches), "Incohérence entre le nombre de lignes et le nombre de batches."

# Reset index (important pour AnnData)
features.reset_index(drop=True, inplace=True)
batches.reset_index(drop=True, inplace=True)

# --- 2. Création de l'AnnData ---

adata = ad.AnnData(X=features.to_numpy(dtype=float))
adata.var_names = features.columns.astype(str)
adata.obs = pd.DataFrame({"ROI": batches})
adata.obs.index = features.index.astype(str)

# --- 3. Application de ComBat ---

sc.pp.combat(adata, key="ROI")

# --- 4. Récupération des données corrigées ---

corrected_df = pd.DataFrame(
    adata.X,
    columns=features.columns,
    index=features.index
)

# On remet la colonne ROI
corrected_df["ROI"] = batches.values

# Sauvegarde
out_path = path_clust_mfi + "df_mfi_tot_corrected.csv"
corrected_df.to_csv(out_path, index=False)
print("✅ Folder with MFI corrected by ComBat:", out_path)

# Aperçu
corrected_df.head()


Dtypes des features :
CD20    float64
CD3     float64
CD4     float64
CD45    float64
CD8     float64
dtype: object
Nombre de NaN par colonne (avant ComBat) :
CD20    0
CD3     0
CD4     0
CD45    0
CD8     0
dtype: int64


AssertionError: Il faut au moins 2 batches (ROI différents) pour appliquer ComBat.

In [ ]:
df_mfi_arcsinh=pd.read_csv(path_clust_mfi+"df_mfi_tot_arcsinh.csv")

# Séparer les features et les batches
features = df_mfi_arcsinh.drop(columns=["ROI"]).copy()
batches = df_mfi_arcsinh["ROI"].astype(str).fillna("NA").copy()

# Vérification minimale
assert batches.nunique() > 1, "At least 2 samples to apply ComBat "
assert len(features) == len(batches), "Number of rows uncorrect"

# Remise à zéro des index (important !)
features.reset_index(drop=True, inplace=True)
batches.reset_index(drop=True, inplace=True)

# Création d'un AnnData
adata = ad.AnnData(X=features.values)
adata.var_names = features.columns
adata.obs = pd.DataFrame({"ROI": batches})
adata.obs.index = features.index.astype(str)


# Application de ComBat
sc.pp.combat(adata, key='ROI')

# Récupération des données corrigées
corrected_df = pd.DataFrame(adata.X, columns=features.columns, index=features.index)
corrected_df["ROI"]=df_mfi["ROI"]
corrected_df.to_csv(path_clust_mfi+"df_mfi_tot_corrected.csv",index=False)
print("✅ Folder with MFI corrected by ComBat: "+path_clust_mfi+"df_mfi_tot_corrected.csv")
corrected_df.head()

## ⚙️ **Standardization**
Standardization rescales features to have zero mean and unit variance, ensuring that all markers contribute equally regardless of their original scale. This prevents variables with large ranges from dominating distance-based analyses such as PCA, clustering, or UMAP.

In [ ]:

# --- Load data ---
df_mfi = pd.read_csv(path_clust_mfi + "df_mfi_tot_raw.csv")
df_mfi=df_mfi.dropna()
df_corrected = pd.read_csv(path_clust_mfi + "df_mfi_tot_corrected.csv")

# --- Basic sanity checks ---
# Ensure the required columns exist
required_meta = ["ROI", "area", "label","centroid-0", "centroid-1"]
missing_in_raw = [c for c in required_meta if c not in df_mfi.columns]
assert "ROI" in df_corrected.columns, "Column 'ROI' missing in df_corrected."
for c in ["area", "centroid-0", "label","centroid-1"]:
    assert c in df_mfi.columns, f"Column '{c}' missing in df_mfi."

# Optional: enforce identical row counts / alignment assumption
assert len(df_mfi) == len(df_corrected), "df_mfi and df_corrected must have the same number of rows."

# --- Standardize only marker columns (exclude 'ROI' and known metadata if present) ---
exclude_cols = set(required_meta) & set(df_corrected.columns)
marker_cols = [c for c in df_corrected.columns if c not in exclude_cols]

scaler = StandardScaler()
X = df_corrected[marker_cols]
X_std = scaler.fit_transform(X)

# Rebuild the standardized frame with original index for safe alignment
corrected_df_std = pd.DataFrame(X_std, columns=marker_cols, index=df_corrected.index)

# --- Reattach labels/metadata ---
corrected_df_std = (
    corrected_df_std
    .assign(
        ROI=df_corrected["ROI"].values,
        area=df_mfi["area"].values,
        **{
            "centroid-0": df_mfi["centroid-0"].values,
            "centroid-1": df_mfi["centroid-1"].values,
            "label": df_mfi["label"].values,
            "Cell_ID": lambda d: d.index.astype(int)
        }
    )
)

# --- Save ---
out_path = path_clust + "mfi_corrected_arcsinh_std.csv"
corrected_df_std.to_csv(out_path, index=False)

# --- Preview ---
print("✅ Folder with MFI corrected by ComBat: "+path_clust_mfi+"df_mfi_corrected_arcsinh_std.csv")
corrected_df_std.head()


✅ Folder with MFI corrected by ComBat: /content/gdrive/MyDrive/these/pipeline/Appendix/clustering/MFI/df_mfi_corrected_arcsinh_std.csv


,CD163,CD20,CD21,CD3,CD31,CD38,CD4,CD45,CD68,CD8,...,ICOS,IFNG,PD1,PanCK,ROI,area,centroid-0,centroid-1,label,Cell_ID
0,-0.910063,0.164367,-1.095329,-1.108856,-0.715654,0.077398,-0.960061,-0.879061,-0.651662,-0.844097,...,-0.923135,-0.494611,-0.366055,-0.314348,ROI,159.0,33.534591,741.943396,2,0
1,-0.680234,1.307160,2.035536,-0.951574,-0.363803,0.525953,-0.818964,-0.412138,-0.593604,-0.531393,...,-0.463683,-0.387398,-0.146279,-0.111590,ROI,248.0,35.709677,643.830645,5,1
2,-0.180980,0.865467,-0.045513,0.578926,-0.239202,0.247858,-0.034866,0.192290,-0.826603,-0.501163,...,0.746984,-0.239852,1.465452,-0.059537,ROI,342.0,34.716374,987.476608,6,2
3,-0.955387,-1.166210,-0.550182,-1.196931,-1.304380,-0.186848,-1.743309,-3.281630,-1.051724,-1.260746,...,0.048755,-0.777125,-0.340668,-0.099867,ROI,135.0,35.918519,153.370370,7,3
4,-0.231985,-1.182756,-0.724659,0.306794,-1.172444,0.225852,-0.041667,-0.621397,-0.463416,-0.761079,...,0.486293,0.523687,2.107784,-0.179020,ROI,208.0,36.475962,269.913462,8,4
